# 01 - Data Preprocessing

Clean, normalize, and prepare all datasets for model training:
1. Map 22 source categories → 12 app categories
2. Clean merchant text for tokenization
3. Augment underrepresented categories with synthetic data
4. Export processed datasets for downstream notebooks

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from preprocessing import (
    load_personal_transactions, load_creditcard, load_synthetic_finance,
    load_budget, CATEGORY_MAP, APP_CATEGORIES, CATEGORY_TO_IDX,
    clean_merchant_text, map_category
)
from data_generator import augment_dataset, generate_synthetic_transactions

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Process Personal Transactions (Auto-Categorization Training Data)

In [ ]:
# Load and map categories
personal = load_personal_transactions('../../dataset/personal_transactions.csv')
print(f'Loaded {len(personal)} transactions')
print(f'\nOriginal → App category mapping:')
for orig, mapped in sorted(CATEGORY_MAP.items()):
    count = (personal['original_category'] == orig).sum()
    print(f'  {orig:25s} → {mapped:15s} ({count} rows)')

personal.head(10)

In [ ]:
# Category distribution after mapping
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

personal['original_category'].value_counts().plot(
    kind='barh', ax=axes[0], color='steelblue'
)
axes[0].set_title('Original Categories (22)')

personal['category'].value_counts().plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title('Mapped App Categories (12)')

plt.tight_layout()
plt.show()

print('\nMapped category counts:')
print(personal['category'].value_counts())

In [ ]:
# Check merchant text cleaning
print('Merchant text cleaning samples:')
for _, row in personal.drop_duplicates('merchant').head(10).iterrows():
    print(f'  "{row["merchant"]}" → "{row["merchant_clean"]}"')

## 2. Augment with Synthetic Data
Balance underrepresented categories (freelance, other, etc.)

In [ ]:
# Tag real data
personal['source'] = 'real'
personal['is_anomaly'] = 0

# Augment to at least 200 samples per category
combined = augment_dataset(personal, target_per_category=200, seed=42)

print(f'Before augmentation: {len(personal)} rows')
print(f'After augmentation:  {len(combined)} rows')
print(f'\nCategory distribution after augmentation:')
print(combined['category'].value_counts().sort_index())
print(f'\nSource breakdown: {combined["source"].value_counts().to_dict()}')

In [ ]:
# Visualize balanced distribution
fig, ax = plt.subplots(figsize=(10, 6))
combined.groupby(['category', 'source']).size().unstack(fill_value=0).plot(
    kind='barh', stacked=True, ax=ax, color=['steelblue', 'lightcoral']
)
ax.set_title('Category Distribution (Real + Augmented)')
ax.set_xlabel('Count')
ax.legend(title='Source')
plt.tight_layout()
plt.show()

## 3. Build Final Auto-Categorization Dataset

In [ ]:
# Select columns needed for auto-categorization training
categorization_df = combined[['merchant_clean', 'category', 'category_idx']].copy()
categorization_df = categorization_df[categorization_df['merchant_clean'].str.len() > 0]

print(f'Auto-categorization dataset: {len(categorization_df)} rows')
print(f'Unique merchants: {categorization_df["merchant_clean"].nunique()}')
print(f'Categories: {categorization_df["category"].nunique()}')

# Save
categorization_df.to_csv('../data/processed/categorization_train.csv', index=False)
print('\nSaved to ../data/processed/categorization_train.csv')
categorization_df.head(10)

## 4. Process Credit Card Data for Anomaly Detection

In [ ]:
creditcard = load_creditcard('../../dataset/creditcard.csv')
print(f'Credit card dataset: {len(creditcard)} rows')
print(f'Fraud cases: {creditcard["is_fraud"].sum()} ({creditcard["is_fraud"].mean()*100:.3f}%)')

# Save processed version
creditcard.to_csv('../data/processed/anomaly_train.csv', index=False)
print('Saved to ../data/processed/anomaly_train.csv')

## 5. Process Synthetic Finance Data for Savings Recommendations

In [ ]:
synthetic = load_synthetic_finance('../../dataset/synthetic_personal_finance_dataset.csv')
print(f'Synthetic finance dataset: {len(synthetic)} rows')

# Feature engineering for savings model
synthetic['expense_ratio'] = synthetic['monthly_expenses_usd'] / synthetic['monthly_income_usd']
synthetic['savings_rate'] = 1 - synthetic['expense_ratio']
synthetic['has_high_debt'] = (synthetic['debt_to_income_ratio'] > 0.4).astype(int)

savings_df = synthetic[[
    'monthly_income_usd', 'monthly_expenses_usd', 'savings_usd',
    'credit_score', 'debt_to_income_ratio', 'expense_ratio',
    'savings_rate', 'has_high_debt', 'has_loan', 'employment_status'
]].copy()

savings_df.to_csv('../data/processed/savings_train.csv', index=False)
print('Saved to ../data/processed/savings_train.csv')
savings_df.describe().round(2)

## 6. Process Budget Data

In [ ]:
budget = load_budget('../../dataset/Budget.csv')

# Aggregate budgets by app category
app_budget = budget.groupby('app_category')['Budget'].sum().reset_index()
app_budget.columns = ['category', 'monthly_budget']

# Add missing categories with 0 budget
for cat in APP_CATEGORIES:
    if cat not in app_budget['category'].values:
        app_budget = pd.concat([
            app_budget,
            pd.DataFrame({'category': [cat], 'monthly_budget': [0]})
        ], ignore_index=True)

app_budget.to_csv('../data/processed/budget_reference.csv', index=False)
print('Budget by app category:')
app_budget.sort_values('monthly_budget', ascending=False)

## 7. Build Time Series Dataset for Forecasting

In [ ]:
# Use personal transactions + synthetic for time series
ts_data = combined[combined['type'] == 'expense'][['date', 'amount']].copy()
ts_data['date'] = pd.to_datetime(ts_data['date'])
daily_totals = ts_data.groupby('date')['amount'].sum().reset_index()
daily_totals.columns = ['date', 'daily_total']

# Fill missing dates with 0
date_range = pd.date_range(daily_totals['date'].min(), daily_totals['date'].max())
daily_totals = daily_totals.set_index('date').reindex(date_range, fill_value=0).reset_index()
daily_totals.columns = ['date', 'daily_total']

daily_totals.to_csv('../data/processed/daily_spending.csv', index=False)
print(f'Daily spending time series: {len(daily_totals)} days')
print(f'Date range: {daily_totals["date"].min()} to {daily_totals["date"].max()}')

daily_totals.set_index('date')['daily_total'].plot(figsize=(14, 4))
plt.title('Daily Spending')
plt.ylabel('Amount ($)')
plt.tight_layout()
plt.show()

## Summary

### Processed Datasets Saved
| File | Rows | Purpose |
|------|------|---------|
| `categorization_train.csv` | ~2400 | Auto-categorization (merchant → category) |
| `anomaly_train.csv` | 284K | Anomaly detection (fraud labels) |
| `savings_train.csv` | 32K | Savings recommendations |
| `budget_reference.csv` | 12 | Budget baseline per category |
| `daily_spending.csv` | ~1095 | Spending forecast time series |

### Next: Notebook 02 — Auto-Categorization Model Training